# ?? ?? ?? ???

- ??: LSTM ???? ?? ????? ?????.
- ??: ??? ??? ?? ???
- ??: ??? ????? ?? ?? ??
- ?? ??: ?? ??? ?? ??? ?? ??? ?? ?????.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import urllib.request
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tqdm

In [ ]:
# urllib.request.urlretrieve("https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt", filename="naver_shopping.txt")

# 데이터 불러오기

In [ ]:
df = pd.read_csv("./data/naver_shopping.txt", sep="\t", header=None, encoding="utf-8-sig")
df.columns = ['score', 'reviews']
print(len(df))
df.head()

In [ ]:
df['label'] = np.select([df.score > 3], [1], default=0)
df.head()

In [ ]:
df_comment = pd.read_csv("./data/total_comment_data.csv", encoding="utf-8-sig")
df_comment

In [ ]:
df_labeled = pd.read_csv("./data/labeled_data/label_comment_df.csv", encoding="utf-8-sig")
df_labeled

# 데이터 중복 제거 및 결측치 처리

In [ ]:
df.drop_duplicates(subset=['reviews'], inplace=True) # Drop duplicate reviews
print(df.isnull().values.any())
print(len(df))

df['reviews'].replace('', np.nan, inplace=True)
print(df.isnull().sum())

In [ ]:
df_comment.drop_duplicates(subset=['comment'], inplace=True) # Drop duplicate reviews
print(df_comment.isnull().values.any())
print(len(df_comment))

df_comment['comment'].replace('', np.nan, inplace=True)
print(df_comment.isnull().sum())

In [ ]:
df_comment.dropna(how='any', inplace=True) # Drop NaN
print(df_comment.isnull().sum())

In [ ]:
df_labeled.drop_duplicates(subset=['comment'], inplace=True) # Drop duplicate reviews
print(df_labeled.isnull().values.any())
print(len(df_labeled))

df_labeled['comment'].replace('', np.nan, inplace=True)
print(df_labeled.isnull().sum())

# 문장 데이터 전처리

In [ ]:
text=''
review=[]
for each_line in df['reviews']:
    review.append(each_line)
#리뷰 잘 들어갔는지 확인
review[2]

In [ ]:

#불용어 제거
import re
def clean_str(text):
    pattern = '[^ㄱ-ㅎㅏ-ㅣ가-힣]'
    text = re.sub(pattern=pattern, repl=' ', string=text)
    pattern = '([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)' # E-mail제거
    text = re.sub(pattern=pattern, repl='', string=text)
    pattern = '(http|ftp|https)://(?:[-\w.]|(?:%[\da-fA-F]{2}))+' # URL제거
    text = re.sub(pattern=pattern, repl='', string=text)
    pattern = '([ㄱ-ㅎㅏ-ㅣ]+)'  # 한글 자음, 모음 제거
    text = re.sub(pattern=pattern, repl='', string=text)
    pattern = '<[^>]*>'         # HTML 태그 제거
    text = re.sub(pattern=pattern, repl='', string=text)
    pattern = '[^\w\s]'         # 특수기호제거
    text = re.sub(pattern=pattern, repl='', string=text)
    return text   

In [ ]:
review_=[]
for i in review:  #문자가 들어있을때는 인덱스 사용하면 안됨!!!!
    a=clean_str(i)
    review_.append(a)  #불용어제거한 review 저장


In [ ]:
review[4] #불용어 제거 전

In [ ]:
review_[4]

In [ ]:
text=''
comment=[]
for each_line in df_comment['comment']:
    comment.append(each_line)
#리뷰 잘 들어갔는지 확인
comment[2]

In [ ]:
comment_=[]
for i in comment:  #문자가 들어있을때는 인덱스 사용하면 안됨!!!!
    a=clean_str(i)
    comment_.append(a)  #불용어제거한 review 저장


In [ ]:
comment[2]

In [ ]:
comment_[2]

In [ ]:
text=''
labeled_comment=[]
for each_line in df_labeled['comment']:
    labeled_comment.append(each_line)
#리뷰 잘 들어갔는지 확인
labeled_comment[2]

In [ ]:
labeled_comment_=[]
for i in labeled_comment:  #문자가 들어있을때는 인덱스 사용하면 안됨!!!!
    a=clean_str(i)
    labeled_comment_.append(a) 

In [ ]:
labeled_comment[2]

In [ ]:
labeled_comment_[2]

# 불용어 처리

In [ ]:
stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

In [ ]:
import nltk
from konlpy.tag import Okt; 
okt=Okt()  #오픈 소스 한국어 분석기

#리뷰하나씩 형태소 추출
a= len(review_) #불용어 제거한 리뷰
token_review_list=[]
for i in tqdm.tqdm(range(0,a)): # 리뷰하나당 처리하기위해 for문 
    token_review=okt.morphs(review_[i])   # morphs=형태소 추출.review_=전처리한 리뷰.
    stopwords_removed_sentence = [word for word in token_review if not word in stopwords] # 불용어 제거
    token_review_list.append(stopwords_removed_sentence) #하나의 리스트를 만들어서 df에 추가해야함.
df['token_review']=token_review_list #형태소단위로 나누어진 리뷰저장
df

In [ ]:
df.to_csv('./data/token_output.csv', encoding='utf-8-sig')

In [ ]:
df = pd.read_csv("./data/token_output.csv", encoding="utf-8-sig")
df

In [ ]:
import nltk
from konlpy.tag import Okt; 
okt=Okt()  #오픈 소스 한국어 분석기

#리뷰하나씩 형태소 추출
a= len(comment_) #불용어 제거한 리뷰
token_review_list=[]
for i in tqdm.tqdm(range(0,a)): # 리뷰하나당 처리하기위해 for문 
    token_review=okt.morphs(comment_[i])   # morphs=형태소 추출.review_=전처리한 리뷰.
    stopwords_removed_sentence = [word for word in token_review if not word in stopwords] # 불용어 제거
    token_review_list.append(stopwords_removed_sentence) #하나의 리스트를 만들어서 df에 추가해야함.
df_comment['token_comment']=token_review_list #형태소단위로 나누어진 리뷰저장
df_comment

In [ ]:
print(df_comment.isnull().values.any())

In [ ]:
df_comment.to_csv('./data/df_comment_tokened.csv', encoding='utf-8-sig')

In [ ]:
df_comment = pd.read_csv('./data/df_comment_tokened.csv')
df_comment

In [ ]:
import nltk
from konlpy.tag import Okt; 
okt=Okt()  #오픈 소스 한국어 분석기

#리뷰하나씩 형태소 추출
a= len(labeled_comment_) #불용어 제거한 리뷰
token_review_list=[]
for i in tqdm.tqdm(range(0,a)): # 리뷰하나당 처리하기위해 for문 
    token_review=okt.morphs(labeled_comment_[i])   # morphs=형태소 추출.review_=전처리한 리뷰.
    stopwords_removed_sentence = [word for word in token_review if not word in stopwords] # 불용어 제거
    token_review_list.append(stopwords_removed_sentence) #하나의 리스트를 만들어서 df에 추가해야함.
df_labeled['token_comment']=token_review_list #형태소단위로 나누어진 리뷰저장
df_labeled

In [ ]:
df_labeled = pd.read_csv("./data/df_labeled_tokened.csv", encoding="utf-8-sig")
df_labeled

In [ ]:
print(df_labeled.isnull().values.any())

In [ ]:
df_labeled.to_csv('./data/df_labeled_tokened.csv', encoding='utf-8-sig')

In [ ]:
train_data, test_data = train_test_split(df, test_size=0.25, random_state=42)
print("Train Reviews : ", len(train_data))
print("Test_Reviews : ", len(test_data))
train_data['label'].value_counts().plot(kind='bar')
print(train_data.groupby('label').size().reset_index(name = 'count'))

In [ ]:
temp_1 = train_data[['token_review']]
temp_1.columns = ['token_comment']
temp_2 = df_comment[['token_comment']]
temp = pd.concat([temp_1, temp_2])
temp.reset_index(inplace=True, drop=True)
temp

In [ ]:
tokenizer_total = Tokenizer()
tokenizer_total.fit_on_texts(temp['token_comment'])

In [ ]:
threshold = 3
total_cnt = len(tokenizer_total.word_index) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in tokenizer_total.word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold - 1, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
# 전체 단어 개수 중 빈도수 2이하인 단어는 제거.
# 0번 패딩 토큰을 고려하여 + 1
vocab_size = total_cnt - rare_cnt + 1
print('단어 집합의 크기 :',vocab_size)

In [ ]:
vocab_size = 87370
tokenizer_total = Tokenizer(vocab_size)
tokenizer_total.fit_on_texts(temp['token_comment'])

In [ ]:
tokenizer_total = Tokenizer(vocab_size)
tokenizer_total.fit_on_texts(temp['token_comment'])
X_train = tokenizer_total.texts_to_sequences(train_data['token_review'])
X_test = tokenizer_total.texts_to_sequences(test_data['token_review'])
X_labeled = tokenizer_total.texts_to_sequences(df_labeled['token_comment'])
X_comment = tokenizer_total.texts_to_sequences(df_comment['token_comment'])

In [ ]:
y_train = np.array(train_data['label'])
y_test = np.array(test_data['label'])
y_labeled = np.array(df_labeled['label'])

In [ ]:
drop_train = [index for index, sentence in enumerate(X_train) if len(sentence) < 1]
# 빈 샘플들을 제거
X_train = np.delete(X_train, drop_train, axis=0)
y_train = np.delete(y_train, drop_train, axis=0)
print(len(X_train))
print(len(y_train))

In [ ]:
print('리뷰의 최대 길이 :',max(len(review) for review in X_train))
print('리뷰의 평균 길이 :',sum(map(len, X_train))/len(X_train))
plt.hist([len(review) for review in X_train], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
print('리뷰의 최대 길이 :',max(len(review) for review in X_comment))
print('리뷰의 평균 길이 :',sum(map(len, X_comment))/len(X_comment))
plt.hist([len(review) for review in X_comment], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
def below_threshold_len(max_len, nested_list):
  count = 0
  for sentence in nested_list:
    if(len(sentence) <= max_len):
        count = count + 1
  print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'%(max_len, (count / len(nested_list))*100))

In [ ]:
max_len = 45
below_threshold_len(max_len, X_train)

In [ ]:
max_len = 125
below_threshold_len(max_len, X_comment)

In [ ]:
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)
X_comment = pad_sequences(X_comment, maxlen=max_len)
X_labeled = pad_sequences(X_labeled, maxlen=max_len)

In [ ]:
X_train.shape, X_test.shape, X_comment.shape, X_labeled.shape

In [ ]:
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
embedding_dim = 100
hidden_units = 128

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(LSTM(hidden_units))
model.add(Dense(1, activation='sigmoid'))

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=4)
mc = ModelCheckpoint('best_model_total.h5', monitor='val_acc', mode='max', verbose=1, save_best_only=True)

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(X_train, y_train, epochs=15, callbacks=[es, mc], batch_size=64, validation_split=0.2)

In [ ]:
loaded_model = load_model('best_model_total.h5')
print("\n 테스트 정확도: %.4f" % (loaded_model.evaluate(X_test, y_test)[1]))

In [ ]:
y_pred = loaded_model.predict(X_test)
y_pred_label = np.round(y_pred.flatten(), 0)
y_pred_label

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
accuracy = accuracy_score(y_test, y_pred_label)
recall = recall_score(y_test, y_pred_label)
precision = precision_score(y_test, y_pred_label)
f1 = f1_score(y_test, y_pred_label)
auc = roc_auc_score(y_test, y_pred_label)
print('accuracy:', accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print("AUC:", auc)

In [ ]:
loaded_model = load_model('best_model_total.h5')
print("\n 테스트 정확도: %.4f" % (loaded_model.evaluate(X_labeled, y_labeled)[1]))

In [ ]:
y_pred = loaded_model.predict(X_labeled)

In [ ]:
y_pred_label = np.round(y_pred.flatten(), 0)
y_pred_label

In [ ]:
from sklearn.metrics import classification_report

# classification_report를 사용하여 다양한 분류 평가 지표 출력
report = classification_report(y_labeled, y_pred_label)
print(report)

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
accuracy = accuracy_score(y_labeled, y_pred_label)
recall = recall_score(y_labeled, y_pred_label)
precision = precision_score(y_labeled, y_pred_label)
f1 = f1_score(y_labeled, y_pred_label)
auc = roc_auc_score(y_labeled, y_pred_label)
print('accuracy:', accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print("AUC:", auc)

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix(y_labeled, y_pred_label)

In [ ]:
df_comment

In [ ]:
loaded_model = load_model('best_model_total.h5')
X_comment = tokenizer_total.texts_to_sequences(df_comment['token_comment'])
X_comment = pad_sequences(X_comment, maxlen=max_len)

In [ ]:
df_labeled

In [ ]:
loaded_model = load_model('best_model_total.h5')
X_labeled = tokenizer_total.texts_to_sequences(df_labeled['token_comment'])
X_labeled = pad_sequences(X_labeled, maxlen=125)

In [ ]:
y_pred = loaded_model.predict(X_labeled)

In [ ]:
df_labeled['감성점수'] = y_pred

In [ ]:
# 행을 랜덤으로 섞기
df_labeled = df_labeled.sample(frac=1).reset_index(drop=True)

In [ ]:
a = df_labeled[((df_labeled['label']==0) & (df_labeled['감성점수']<=0.5)) | ((df_labeled['label']==1) & (df_labeled['감성점수']>0.5))]
a = a.sample(frac=1).reset_index(drop=True).loc[:,['comment','label', '감성점수']]
a

In [ ]:
a.loc[100:3166, ['comment', '감성점수']]

In [ ]:
a[a['label'] == 0].head()

In [ ]:
df_labeled.iloc[2264:2340, :]

In [ ]:
df_comment.to_csv('./data/df_comment_score.csv', encoding='utf-8-sig')

In [ ]:
df_comment.info()

In [ ]:
df_frame = pd.read_excel('./data/mater_table.xlsx')
df_frame['댓글 수'] = np.nan
df_frame['영상 수'] = np.nan
df_frame['감성점수'] = np.nan
df_frame

In [ ]:
total_year_video_df = pd.read_csv('./data/total_year_video_df.csv')
total_year_video_df

In [ ]:
df_comment['weighted score'] = df_comment['감성점수'] * (df_comment['like_num'] + 1)
df_comment

In [ ]:
df_comment['like_num'].sum() + len(df_comment)

In [ ]:
for name in total_year_video_df['채널명'].unique():
    print(list(total_year_video_df['채널명'].unique()).index(name), name)
    temp_total = total_year_video_df[total_year_video_df['채널명'] == name]
    temp_total.reset_index(drop=True, inplace=True)
    for date in df_frame['월'].unique():
        year, month = map(int, date.split('/'))
        num_list = temp_total[(temp_total['업데이트 날짜'].str.split('-').str[0].astype(int) == year) & (temp_total['업데이트 날짜'].str.split('-').str[1].astype(int) == month)].index
        temp_month = pd.DataFrame(columns=df_comment.columns)
        for i in num_list:
            temp = df_comment[(df_comment['owner_channel'] == name) & (df_comment['video_num'] == i)]
            temp_month = pd.concat([temp_month, temp])
        temp_month['weighted score'] = temp_month['감성점수'] * (temp_month['like_num'] + 1)
        temp_frequency = pd.DataFrame(temp_month.groupby('youtube_id').count()['comment'])
        df_frame.loc[(df_frame['월'] == date) & (df_frame['유투버'] == name), '댓글 수'] = len(temp_month)
        df_frame.loc[(df_frame['월'] == date) & (df_frame['유투버'] == name), '영상 수'] = len(num_list)
        if len(temp_month) == 0:
            df_frame.loc[(df_frame['월'] == date) & (df_frame['유투버'] == name), '감성점수'] = np.nan
            
            
        else:
            df_frame.loc[(df_frame['월'] == date) & (df_frame['유투버'] == name), '감성점수'] = temp_month['weighted score'].sum() / (temp_month['like_num'].sum() + len(temp_month))
            # print(temp_month['weighted score'].sum() / (temp_month['like_num'].sum() + len(temp_month)))

In [ ]:
df_frame.to_csv('./data/감성점수.csv', encoding='utf-8-sig', index=False)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_data['token_review'])

In [ ]:
print(tokenizer.word_index)

In [ ]:
threshold = 3
total_cnt = len(tokenizer.word_index) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in tokenizer.word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold - 1, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
# 전체 단어 개수 중 빈도수 2이하인 단어는 제거.
# 0번 패딩 토큰을 고려하여 + 1
vocab_size = total_cnt - rare_cnt + 1
print('단어 집합의 크기 :',vocab_size)

In [ ]:
tokenizer_comment = Tokenizer()
tokenizer_comment.fit_on_texts(df_comment['token_comment'])

In [ ]:
print(tokenizer_comment.word_index)

In [ ]:
threshold = 3
total_cnt = len(tokenizer_comment.word_index) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in tokenizer_comment.word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold - 1, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
# 전체 단어 개수 중 빈도수 2이하인 단어는 제거.
# 0번 패딩 토큰을 고려하여 + 1
vocab_size = total_cnt - rare_cnt + 1
print('단어 집합의 크기 :',vocab_size)

In [ ]:
tokenizer = Tokenizer(vocab_size) 
tokenizer.fit_on_texts(train_data['token_review'])
X_train = tokenizer.texts_to_sequences(train_data['token_review'])
X_test = tokenizer.texts_to_sequences(test_data['token_review'])
X_comment = tokenizer.texts_to_sequences(df_labeled['token_comment'])

In [ ]:
y_train = np.array(train_data['label'])
y_test = np.array(test_data['label'])
y_comment = np.array(df_labeled['label'])

In [ ]:
drop_train = [index for index, sentence in enumerate(X_train) if len(sentence) < 1]

In [ ]:
# 빈 샘플들을 제거
X_train = np.delete(X_train, drop_train, axis=0)
y_train = np.delete(y_train, drop_train, axis=0)
print(len(X_train))
print(len(y_train))

In [ ]:
print('리뷰의 최대 길이 :',max(len(review) for review in X_train))
print('리뷰의 평균 길이 :',sum(map(len, X_train))/len(X_train))
plt.hist([len(review) for review in X_train], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
def below_threshold_len(max_len, nested_list):
  count = 0
  for sentence in nested_list:
    if(len(sentence) <= max_len):
        count = count + 1
  print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'%(max_len, (count / len(nested_list))*100))

In [ ]:
max_len = 45
below_threshold_len(max_len, X_train)

In [ ]:
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)
X_comment = pad_sequences(X_comment, maxlen=max_len)

In [ ]:
X_train.shape, X_test.shape, X_comment.shape

In [ ]:
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
embedding_dim = 100
hidden_units = 128

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(LSTM(hidden_units))
model.add(Dense(1, activation='sigmoid'))

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=4)
mc = ModelCheckpoint('best_model.h5', monitor='val_acc', mode='max', verbose=1, save_best_only=True)

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(X_train, y_train, epochs=15, callbacks=[es, mc], batch_size=64, validation_split=0.2)

In [ ]:
loaded_model = load_model('best_model.h5')
print("\n 테스트 정확도: %.4f" % (loaded_model.evaluate(X_test, y_test)[1]))

In [ ]:
loaded_model = load_model('best_model.h5')
loaded_model.summary()

In [ ]:
y_pred = loaded_model.predict(X_test)
y_pred_label = np.round(y_pred.flatten(), 0)
y_pred_label

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
accuracy = accuracy_score(y_test, y_pred_label)
recall = recall_score(y_test, y_pred_label)
precision = precision_score(y_test, y_pred_label)
f1 = f1_score(y_test, y_pred_label)
auc = roc_auc_score(y_test, y_pred_label)
print('accuracy:', accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print("AUC:", auc)

In [ ]:
class_weights = {0: 10, 1: 1}

In [ ]:
loaded_model = load_model('best_model.h5')
print("\n 테스트 정확도: %.4f" % (loaded_model.evaluate(X_comment, y_comment)[1]))

In [ ]:
y_pred = loaded_model.predict(X_comment)

In [ ]:
y_pred_label = np.round(y_pred.flatten(), 0)
y_pred_label

In [ ]:
from sklearn.metrics import classification_report

# classification_report를 사용하여 다양한 분류 평가 지표 출력
report = classification_report(y_comment, y_pred_label)
print(report)

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
accuracy = accuracy_score(y_comment, y_pred_label)
recall = recall_score(y_comment, y_pred_label)
precision = precision_score(y_comment, y_pred_label)
f1 = f1_score(y_comment, y_pred_label)
auc = roc_auc_score(y_comment, y_pred_label)
print('accuracy:', accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print("AUC:", auc)

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix(y_comment, y_pred_label)

In [ ]:
import pickle
with open('tokenizer.pickle', 'wb') as handle:
     pickle.dump(tokenizer, handle)

with open('tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)